# GrievX YOLO26 Rebuild Notebook

This notebook rebuilds the GrievX classifier from a clean workspace using the new real-world video data.

Goal:
- remove stale model artifacts
- rebuild the dataset from video frames
- train a stronger YOLO classification model
- evaluate validation performance
- run inference on real sample videos
- save the best versioned weights

Important:
- No model can be guaranteed to reach 100% accuracy.
- This notebook is tuned for the highest practical accuracy by using a larger model, stronger training settings, and cleaner dataset handling.


## 1. Cleanup Existing YOLO Artifacts

Start from a clean state by removing old runs, weights, caches, logs, and stale generated outputs. Keep the raw source videos and frame folders because they are needed to rebuild the dataset.


In [ ]:
from pathlib import Path
import shutil
import os
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()

paths_to_remove = [
    PROJECT_ROOT / "runs",
    PROJECT_ROOT / "dataset_grievx",
    PROJECT_ROOT / "__pycache__",
]

for pattern in ("*.pt", "*.onnx", "*.engine", "*.cache", "*.log"):
    for file_path in PROJECT_ROOT.glob(pattern):
        paths_to_remove.append(file_path)

for path in paths_to_remove:
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    elif path.is_file():
        try:
            path.unlink()
        except FileNotFoundError:
            pass

print("Cleanup complete.")
print("Removed generated runs, dataset copies, caches, and model artifacts if present.")

## 2. Import Libraries and Set Paths

Define the project paths, class names, and the high-accuracy training defaults used throughout the notebook.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import torch
import yaml

try:
    from ultralytics import YOLO
except Exception as exc:
    raise SystemExit("Install dependencies first: pip install -r requirements.txt") from exc

PROJECT_ROOT = Path.cwd()
RAW_VIDEO_ROOT = PROJECT_ROOT / "Video_Dataset"
FRAMES_ROOT = PROJECT_ROOT / "frames_output"
DATASET_ROOT = PROJECT_ROOT / "dataset_grievx"
RUNS_ROOT = PROJECT_ROOT / "runs"
EXPORT_ROOT = PROJECT_ROOT / "exports"
CONFIG_PATH = PROJECT_ROOT / "grievx_config.yaml"
TRAIN_SCRIPT = PROJECT_ROOT / "prepare_grievx_dataset.py"
PREDICT_SCRIPT = PROJECT_ROOT / "predict_grievx_video.py"

CLASSES = [
    "Crop_Damage",
    "Drainage_issue",
    "Drinking_water_shortage",
    "Industrial_Pollution_complaint",
    "Road_Damage",
]

MODEL_VARIANT = "yolo26m-cls.pt"
TRAIN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_IMGSZ = 384 if TRAIN_DEVICE == "cuda" else 320
TRAIN_BATCH = 16 if TRAIN_DEVICE == "cuda" else 8
TRAIN_EPOCHS = 60 if TRAIN_DEVICE == "cuda" else 30
TRAIN_PATIENCE = 15
TRAIN_NAME = "grievx_yolo26_highacc"

print("Project root:", PROJECT_ROOT)
print("Device:", TRAIN_DEVICE)
print("Model variant:", MODEL_VARIANT)
print("Image size:", TRAIN_IMGSZ)
print("Batch size:", TRAIN_BATCH)
print("Epochs:", TRAIN_EPOCHS)

## 3. Configure Dataset for New Real-Time Classes

Use the new real-world complaint classes and rebuild the dataset from the extracted video frames. This keeps the label mapping aligned with the complaint categories.


In [ ]:
print("Source frame folders:")
for class_name in CLASSES:
    frame_dir = FRAMES_ROOT / class_name
    video_dir = RAW_VIDEO_ROOT / class_name
    print(f"  {class_name}:")
    print(f"    frames: {frame_dir} {'OK' if frame_dir.is_dir() else 'MISSING'}")
    print(f"    videos: {video_dir} {'OK' if video_dir.is_dir() else 'MISSING'}")

print()
print("Rebuilding dataset from frame folders...")
cmd = [sys.executable, str(TRAIN_SCRIPT), "--clean"]
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(f"Dataset rebuild failed with exit code {result.returncode}")

print("Dataset rebuild completed.")

## 4. Validate and Organize Train/Validation/Test Splits

Check dataset integrity, verify that the split is balanced by class, and inspect class counts before training. For classification, the notebook keeps train and validation splits for model selection and reports a holdout-style quality check on unseen sample videos during evaluation.


In [ ]:
import json
from collections import defaultdict

split_manifest = DATASET_ROOT / "split_manifest.json"
if not split_manifest.is_file():
    raise FileNotFoundError("dataset split manifest not found; rerun the rebuild cell first")

with open(split_manifest, encoding="utf-8") as f:
    manifest = json.load(f)

print("Split summary:")
for split_name in ("train", "val"):
    print(f"\n{split_name.upper()}")
    counts = manifest.get("counts", {}).get(split_name, {})
    for class_name in CLASSES:
        print(f"  {class_name}: {counts.get(class_name, 0)} images")

train_total = sum(manifest.get("counts", {}).get("train", {}).values())
val_total = sum(manifest.get("counts", {}).get("val", {}).values())
print(f"\nTotal train images: {train_total}")
print(f"Total val images: {val_total}")

# Basic duplicate / corruption check
corrupt = []
for image_path in DATASET_ROOT.rglob("*.jpg"):
    try:
        _ = image_path.stat().st_size
    except Exception:
        corrupt.append(image_path)

print(f"Corrupt files found: {len(corrupt)}")
if corrupt:
    for item in corrupt[:10]:
        print("  ", item)

print("Train/validation structure is ready.")

## 5. Build or Load YOLO Model Configuration

Load the larger YOLO26 classification checkpoint and prepare the training settings for the highest practical accuracy.


In [ ]:
best_model = YOLO(MODEL_VARIANT)
print("Base model loaded:", MODEL_VARIANT)
print("Training output name:", TRAIN_NAME)
print("Training device:", TRAIN_DEVICE)
print("Training image size:", TRAIN_IMGSZ)
print("Training epochs:", TRAIN_EPOCHS)
print("Training batch:", TRAIN_BATCH)
print("Training patience:", TRAIN_PATIENCE)

## 6. Train the Model with Hyperparameter Tuning

Train the classification model with a stronger preset: larger checkpoint, higher image size, AdamW, label smoothing, and early stopping.


In [ ]:
train_results = best_model.train(
    data=str(DATASET_ROOT),
    epochs=TRAIN_EPOCHS,
    imgsz=TRAIN_IMGSZ,
    batch=TRAIN_BATCH,
    patience=TRAIN_PATIENCE,
    device=TRAIN_DEVICE,
    optimizer="AdamW",
    lr0=0.003,
    lrf=0.05,
    weight_decay=0.0005,
    label_smoothing=0.05,
    dropout=0.1,
    cos_lr=True,
    seed=42,
    deterministic=True,
    workers=4,
    project=str(RUNS_ROOT / "classify"),
    name=TRAIN_NAME,
    exist_ok=True,
    pretrained=True,
    verbose=True,
    plots=True,
    save_period=5,
)

best_weights = RUNS_ROOT / "classify" / TRAIN_NAME / "weights" / "best.pt"
print("Training complete.")
print("Best weights:", best_weights)
print("Results folder:", RUNS_ROOT / "classify" / TRAIN_NAME)

## 7. Evaluate Accuracy and Class Performance

Run validation to inspect top-1 accuracy, top-5 accuracy, and the validation run artifacts such as the confusion matrix.


In [ ]:
trained_model = YOLO(str(best_weights))
val_results = trained_model.val(
    data=str(DATASET_ROOT),
    imgsz=TRAIN_IMGSZ,
    device=TRAIN_DEVICE,
)

print("Validation results object:")
print(val_results)
print()
print("Results dictionary:")
if hasattr(val_results, "results_dict"):
    print(val_results.results_dict)

print()
print("Top-1:", getattr(val_results, "top1", "n/a"))
print("Top-5:", getattr(val_results, "top5", "n/a"))
print("Confusion matrix and validation plots are saved inside the run directory.")

## 8. Run Inference on Real-Time Video Samples

Test the retrained model on a real sample video and verify the predicted complaint label and category.


In [ ]:
EXAMPLE_VIDEO = RAW_VIDEO_ROOT / "Crop_Damage" / "crop damage.mp4"
if not EXAMPLE_VIDEO.is_file():
    print("Example video not found:", EXAMPLE_VIDEO)
else:
    command = [
        sys.executable,
        str(PREDICT_SCRIPT),
        "--video",
        str(EXAMPLE_VIDEO),
        "--model",
        str(best_weights),
        "--max-frames",
        "30",
    ]
    result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise SystemExit(f"Prediction failed with exit code {result.returncode}")

## 9. Save, Export, and Version the Best Model

Copy the best checkpoint into a versioned export folder and try exporting to ONNX for deployment.


In [ ]:
version_dir = EXPORT_ROOT / TRAIN_NAME
version_dir.mkdir(parents=True, exist_ok=True)

trained_model = YOLO(str(best_weights))

if best_weights.is_file():
    copied_weights = version_dir / "best.pt"
    shutil.copy2(best_weights, copied_weights)
    print("Saved versioned checkpoint:", copied_weights)
else:
    print("Best weights not found yet. Train the model first.")

metadata = {
    "model_variant": MODEL_VARIANT,
    "train_name": TRAIN_NAME,
    "device": TRAIN_DEVICE,
    "imgsz": TRAIN_IMGSZ,
    "batch": TRAIN_BATCH,
    "epochs": TRAIN_EPOCHS,
    "patience": TRAIN_PATIENCE,
    "classes": CLASSES,
}
with open(version_dir / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print("Saved metadata:", version_dir / "training_metadata.json")

try:
    exported_onnx = trained_model.export(format="onnx", imgsz=TRAIN_IMGSZ)
    print("ONNX export:", exported_onnx)
except Exception as exc:
    print("ONNX export skipped or failed:", exc)